In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt

import os
from pathlib import Path
import datetime
import warnings

from vivarium_helpers.utils import convert_to_categorical, constant_categorical, print_memory_usage, Timer

from vivarium_helpers.projects.alzheimers import loading, population
from vivarium_helpers.projects.alzheimers.population import RunType
from vivarium_helpers.projects.alzheimers.results import AlzheimersResultsProcessor

!date
!whoami
!pwd

Thu Nov 19 20:56:57 PST 2026


lutzes


/ihme/homes/lutzes/vivarium_research_alzheimers/results_tables


# Record version of Vivarium Helpers

In [3]:
!conda list | grep vivarium

# packages in environment at /ihme/homes/lutzes/.conda/envs/vivarium_csu_alzheimers_artifact:
vivarium-artifact         1.1.1                    pypi_0    pypi
vivarium-build-utils      4.11.1                   pypi_0    pypi
vivarium-cluster-tools    4.8.0                    pypi_0    pypi
vivarium-config-tree      5.2.1                    pypi_0    pypi
vivarium-csu-alzheimers   0.1.dev76+g982fe3fb1          pypi_0    pypi
vivarium-dependencies     1.3.1                    pypi_0    pypi
vivarium-engine           6.0.1                    pypi_0    pypi
vivarium-gbd-access       7.0.1                    pypi_0    pypi
vivarium-gbd-mapping      6.0.7                    pypi_0    pypi
vivarium-helpers          0.5.0.dev251111          pypi_0    pypi
vivarium-inputs           9.1.3                    pypi_0    pypi
vivarium-public-health    6.8.0                    pypi_0    pypi
vivarium-risk-distributions 3.2.1                    pypi_0    pypi


In [4]:
!pip freeze | grep vivarium

vivarium-artifact==1.1.1
vivarium-build-utils==4.11.1
vivarium-cluster-tools==4.8.0
vivarium-config-tree==5.2.1
-e git+https://github.com/ihmeuw/vivarium_csu_alzheimers.git@982fe3fb1cfe03824fcf5cf945850e1c9b0688b8#egg=vivarium_csu_alzheimers
vivarium-dependencies==1.3.1
vivarium-engine==6.0.1
vivarium-gbd-mapping==6.0.7
vivarium-public-health==6.8.0
vivarium-risk-distributions==3.2.1
vivarium_gbd_access==7.0.1
-e git+https://github.com/ihmeuw/vivarium_helpers.git@5b35fe7c0e45dd7296841aa977c3d299d1fdcaac#egg=vivarium_helpers
vivarium_inputs==9.1.3


# Define data directories

In [5]:
# Project directory
project_dir = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/')

# Output directory to store final results, in a folder named with
# today's date. These files include draws. write_final_csvs.ipynb writes
# the client files without draws to the no_draws folder next to it.
results_date = datetime.date.today().strftime('%Y_%m_%d')
output_dir = project_dir / 'results' / 'final_csvs' / results_date / 'draws'
output_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df, filename):
    """Write df to output_dir, with a warning if the file already exists."""
    path = output_dir / filename
    if path.exists():
        warnings.warn(f"Overwriting existing file {path}")
    df.to_csv(path, index=False)


# Output from multistate life table
# mslt_output_dir = output_dir.parent / 'results_11_14_2025_mslt'
mslt_output_dir = project_dir / 'results' / 'mslt_draws'
print(mslt_output_dir.exists())
output_dir

True


PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/final_csvs/2026_11_19/draws')

In [6]:
# # For testing: Run directory containing model 8.3 results for all
# # locations
# model_run_subdir = 'results/abie_consistent_model_test/united_states_of_america/2025_10_28_08_55_05/'

# # Results directory for model 8.3, for testing
# results_dirs = project_dir / model_run_subdir / 'results/'

# Artifact for models 10.0 - 12.2 was '10.0'
# Artifact for models 13.0 and 14.0
artifact_model_number = '13.0'


In [7]:
# Model 8.4 results (final runs for 10/31/25 intermediate results)
# Each batch run contains all locations, 100 random seeds, and 3 or 4 draws
model_run_dir_8_4 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model8.4/model_spec')

# Model 8.7 results (updated final run completed on 11/10/2025)
model_run_dir_8_7 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model8.7/model_spec')

# Model 12.2 results (updated final run completed on 02/14/2026)
model_run_dir_12_2 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model12.2')

# Model 13.0 results (rebuilt artifacts, run started 10/29/2026)
# This run has a single run directory. The loader adds the 'results'
# subfolder to each run directory.
# The model 13.0 run is V&V sized (5 seeds of 20,000 simulants) and has
# only 3 locations, so use the V&V run type for the model scale.
run_type_13_0 = RunType.V_AND_V
locations_13_0 = ['United States of America', 'China', 'Brazil']
model_run_dir_13_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model13.0/model_spec/2026_10_29_14_47_34')

# Model 14.0 results (new testing ramps and scenarios, run started 10/30/2026)
# Same artifacts as model 13.0. This run is also V&V sized (5 seeds of
# 20,000 simulants), with all 10 locations.
run_type_14_0 = RunType.V_AND_V
locations_14_0 = loading.LOCATIONS
model_run_dir_14_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model14.0/model_spec/2026_10_30_13_30_29')

# Model 16.0 results (production run started 11/18/2026)
# Same artifacts as model 13.0 and the same scenarios as model 14.0.
# This run is production sized (100 seeds of 20,000 simulants), with all
# 10 locations and 25 draws.
run_type_16_0 = RunType.FINAL
locations_16_0 = loading.LOCATIONS
model_run_dir_16_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08')

# Batch runs are stored in "timestamp" subdirectories of the model run
# directory.
# In model 12.2, the run directories are nested under two more levels of
# subfolders: batch number, then model_spec, then timestamp.

# Select a model and filter for testing
# batch_run_dirs = loading.list_batch_run_directories(model_run_dir_12_2)#[:2]
# batch_run_dirs = [str(model_run_dir_13_0)]
# batch_run_dirs = [str(model_run_dir_14_0)]
batch_run_dirs = [str(model_run_dir_16_0)]

batch_run_dirs

['/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08']

# Create an AlzheimersResultsProcessor object to process results

In [8]:
results = AlzheimersResultsProcessor(
    artifact_model_number,
    run_type_16_0,
    locations=locations_16_0,
    batch_run_dirs=batch_run_dirs,
)

In [9]:
results.initial_simulation_population

2000000

In [10]:
results.locations

['United States of America',
 'Brazil',
 'China',
 'Germany',
 'Israel',
 'Japan',
 'Spain',
 'Sweden',
 'Taiwan',
 'United Kingdom']

In [11]:
results.draws

[202,
 170,
 164,
 135,
 243,
 181,
 197,
 35,
 11,
 75,
 41,
 26,
 216,
 105,
 15,
 198,
 233,
 179,
 19,
 156,
 172,
 90,
 96,
 177,
 24]

In [12]:
len(set(results.draws))

25

# Load population data and calculate model scale

In [13]:
with Timer():
    results.load_population_data(append_aggregates=False)
results.ops.describe(results.model_scale)

Elapsed time: 0:00:14.679815


,count,mean,std,min,25%,50%,75%,max
location,,,,,,,,
Brazil,25.0,0.666241,0.048702,0.569066,0.647484,0.664228,0.697841,0.746260
China,25.0,0.074932,0.004991,0.066343,0.071189,0.073947,0.080082,0.082505
Germany,25.0,0.564469,0.046950,0.483817,0.541823,0.565487,0.574408,0.692730
Israel,25.0,26.437376,2.594453,21.371444,25.341007,26.388861,27.918803,33.464651
Japan,25.0,0.369508,0.022974,0.323476,0.355148,0.364880,0.390183,0.409644
Spain,25.0,1.898532,0.096439,1.775293,1.819973,1.873009,1.940022,2.072102
Sweden,25.0,7.868337,0.443373,7.053982,7.624894,7.863023,8.151224,8.754205
Taiwan,25.0,5.241038,0.277182,4.550281,5.137899,5.242813,5.410650,5.660475
United Kingdom,25.0,1.270413,0.060722,1.121093,1.232772,1.271746,1.314122,1.389850


In [14]:
results.person_time

,location,sex,event_year,input_draw,value,age_group
0,United States of America,Female,2025,202,1.128934e+07,25_to_29
1,United States of America,Female,2025,170,1.033839e+07,25_to_29
2,United States of America,Female,2025,164,1.120331e+07,25_to_29
3,United States of America,Female,2025,135,1.001506e+07,25_to_29
4,United States of America,Female,2025,243,1.131052e+07,25_to_29
...,...,...,...,...,...,...
569995,United Kingdom,Male,2100,172,1.325974e+05,95_plus
569996,United Kingdom,Male,2100,90,1.267402e+05,95_plus
569997,United Kingdom,Male,2100,96,1.354910e+05,95_plus
569998,United Kingdom,Male,2100,177,1.356587e+05,95_plus


# Load prevalence and append counterfactual person-time

This calculates counterfactual person-time in treatment scenario for correct rate calculation.

In [15]:
with Timer():
    results.load_prevalence_and_append_counterfactual_person_time(append_aggregates=True)
results.person_time

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


36867.612507 MB after read_parquet


234.618534 MB after aggregating seeds and converting dtypes
234.618534 MB total
234.618534 MB after enforcing dtypes
Elapsed time: 912.1832 seconds


223.448606 MB measure


268.136474 MB minuend
44.696474 MB subtrahend


191.529254 MB minuend re-indexed
31.929254 MB subtrahend re-indexed


... (2 outputs left out)

/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Elapsed time: 0:15:35.257087


,location,sex,event_year,input_draw,value,age_group,scenario
0,United States of America,Female,2025,202,1.063863e+07,65_to_69,baseline
1,United States of America,Female,2025,170,9.662163e+06,65_to_69,baseline
2,United States of America,Female,2025,164,1.057504e+07,65_to_69,baseline
3,United States of America,Female,2025,135,9.369922e+06,65_to_69,baseline
4,United States of America,Female,2025,243,1.067762e+07,65_to_69,baseline
...,...,...,...,...,...,...,...
6041995,United States of America,Both,2100,243,8.062193e+07,65_plus,baseline
6041996,United States of America,Both,2100,243,8.071912e+07,65_plus,intermediate_testing_with_treatment
6041997,United States of America,Both,2100,243,8.062193e+07,65_plus,intermediate_testing_zero_impact_treatment
6041998,United States of America,Both,2100,243,8.071912e+07,65_plus,slow_testing_with_treatment


# DALYs and averted DALYs

In [16]:
filename = 'ylls'
short_name = filename

with Timer():
    results.load_measure_from_batch_runs(filename, short_name, filters=loading.FINAL_RESULTS_FILTERS[filename])

print(len(results.data[short_name]), 'rows')
results.data[short_name].head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025'), ('entity', '==', 'alzheimers_disease_state')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


12289.212426 MB after read_parquet


74.486199 MB after aggregating seeds and converting dtypes
74.486199 MB total
74.486199 MB after enforcing dtypes
Elapsed time: 801.2126 seconds
Elapsed time: 0:13:21.212876
3724000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,value,location
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,ylls,baseline,Female,0.0,Japan
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,ylls,baseline,Male,0.0,Japan
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,ylls,slow_testing_zero_impact_treatment,Female,0.0,Japan
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,ylls,slow_testing_zero_impact_treatment,Male,0.0,Japan
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,ylls,intermediate_testing_zero_impact_treatment,Female,0.0,Japan


In [17]:
filename = 'ylds'
short_name = filename

with Timer():
    results.load_measure_from_batch_runs(filename, short_name, filters=loading.FINAL_RESULTS_FILTERS[filename])

print(len(results.data[short_name]), 'rows')
results.data[short_name].head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025'), ('entity', '==', 'alzheimers_disease_and_other_dementias')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


36867.613275 MB after read_parquet


234.619302 MB after aggregating seeds and converting dtypes
234.619302 MB total
234.619302 MB after enforcing dtypes
Elapsed time: 872.2928 seconds
Elapsed time: 0:14:32.293088
11172000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,ylds,baseline,Female,alzheimers_blood_based_biomarker_state,0.0,Japan
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,ylds,baseline,Female,alzheimers_disease_state,0.0,Japan
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,ylds,baseline,Female,alzheimers_mild_cognitive_impairment_state,0.0,Japan
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,ylds,baseline,Male,alzheimers_blood_based_biomarker_state,0.0,Japan
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,ylds,baseline,Male,alzheimers_disease_state,0.0,Japan


In [18]:
dalys = results.process_dalys(results.data.ylls, results.data.ylds)
print_memory_usage(dalys)
print(len(dalys), 'rows')
dalys.tail()

/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:355: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace({'entity': {'alzheimers_disease_state': 'AD'}})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:375: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace({'entity': {'alzheimers_disease_and_other_dementias': 'AD'}})


212.276394 MB measure


43.100262 MB minuend
258.560262 MB subtrahend


30.332983 MB minuend re-indexed
181.952983 MB subtrahend re-indexed


181.953047 MB difference


181.952394 MB difference with reset index
191.529314 MB final difference


435.717056 MB 
20748000 rows


,age_group,artifact_path,entity,entity_type,event_year,measure,sex,sub_entity,input_draw,scenario,location,value,metric,disease_stage
20747995,95_plus,/mnt/team/simulation_science/pub/models/vivari...,AD,cause,2100,Averted DALYs Associated with AD,Female,alzheimers_disease_state,243,accelerated_testing_with_treatment,Israel,-2428.165853,Number,alzheimers_disease_state
20747996,95_plus,/mnt/team/simulation_science/pub/models/vivari...,AD,cause,2100,Averted DALYs Associated with AD,Female,alzheimers_mild_cognitive_impairment_state,243,accelerated_testing_with_treatment,Israel,-35.311289,Number,alzheimers_mild_cognitive_impairment_state
20747997,95_plus,/mnt/team/simulation_science/pub/models/vivari...,AD,cause,2100,Averted DALYs Associated with AD,Male,alzheimers_blood_based_biomarker_state,243,accelerated_testing_with_treatment,Israel,0.000000,Number,alzheimers_blood_based_biomarker_state
20747998,95_plus,/mnt/team/simulation_science/pub/models/vivari...,AD,cause,2100,Averted DALYs Associated with AD,Male,alzheimers_disease_state,243,accelerated_testing_with_treatment,Israel,-1694.485422,Number,alzheimers_disease_state
20747999,95_plus,/mnt/team/simulation_science/pub/models/vivari...,AD,cause,2100,Averted DALYs Associated with AD,Male,alzheimers_mild_cognitive_impairment_state,243,accelerated_testing_with_treatment,Israel,-21.136418,Number,alzheimers_mild_cognitive_impairment_state


In [19]:
dalys.query("0 < abs(value) < 1e-8").measure.unique()

[], Categories (2, object): ['Averted DALYs Associated with AD', 'DALYs Associated with AD']

In [20]:
dalys.query("0 < abs(value) < 1e-8").value.describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: value, dtype: float64

In [21]:
dalys.query("abs(value) >= 1e-8").value.describe()

count    7.806271e+06
mean     3.302611e+04
std      7.877132e+04
min     -7.604196e+03
25%      7.995299e+01
50%      1.169817e+03
75%      1.526821e+04
max      7.489468e+05
Name: value, dtype: float64

In [22]:
# 5m 22s to run on first 3 batches of 8.4 (10 draws)
with Timer():
    # dalys_output = results.summarize_and_beautify(dalys)
    # process_* returns event_year as an unordered categorical, and the
    # event_year >= 2025 filter in do_final_processing cannot compare it
    # with the installed pandas version, so cast it back to int16 first
    dalys_output = results.do_final_processing(dalys.astype({'event_year': 'int16'}), include_draws=True)
print_memory_usage(dalys_output)
print(len(dalys_output), 'rows')
dalys_output.tail()

2026-11-19 21:41:42.975149


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 1253.9443 seconds


2026-11-19 22:04:22.487824
Elapsed time: 0:22:39.620212
618.889388 MB 
2667600 rows


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:994: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
2667595,2100,Israel,65_plus,Male,MCI due to AD,"Intermediate Testing, With Treatment",DALYs Associated with AD,Rate,0.000356,0.000225,...,0.000368,0.000219,0.000301,0.000355,0.000417,0.000266,0.000358,0.000267,0.000475,0.000375
2667596,2100,Israel,65_plus,Both,MCI due to AD,"Intermediate Testing, With Treatment",DALYs Associated with AD,Rate,0.000467,0.000292,...,0.000527,0.000289,0.000382,0.000463,0.000537,0.000327,0.000470,0.000373,0.000568,0.000498
2667597,2100,Israel,65_plus,Female,MCI due to AD,"Accelerated Testing, With Treatment",DALYs Associated with AD,Rate,0.000565,0.000352,...,0.000668,0.000350,0.000454,0.000558,0.000644,0.000381,0.000569,0.000467,0.000650,0.000608
2667598,2100,Israel,65_plus,Male,MCI due to AD,"Accelerated Testing, With Treatment",DALYs Associated with AD,Rate,0.000356,0.000225,...,0.000368,0.000219,0.000301,0.000355,0.000417,0.000266,0.000358,0.000267,0.000475,0.000375
2667599,2100,Israel,65_plus,Both,MCI due to AD,"Accelerated Testing, With Treatment",DALYs Associated with AD,Rate,0.000467,0.000292,...,0.000527,0.000289,0.000382,0.000463,0.000537,0.000327,0.000470,0.000373,0.000568,0.000498


In [23]:
save_csv(dalys_output, 'dalys.csv')

/tmp/ipykernel_1684666/62930288.py:16: UserWarning: Overwriting existing file /mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/final_csvs/2026_11_19/draws/dalys.csv
  warnings.warn(f"Overwriting existing file {path}")


# Incidence

In [24]:
filename = 'counts_new_simulants'
short_name = 'incidence_bbbm'

with Timer():
    results.load_measure_from_batch_runs(filename, short_name, filters=loading.FINAL_RESULTS_FILTERS[filename])

print(len(results.data[short_name]), 'rows')
results.data[short_name].head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


10799.611571 MB after read_parquet


63.313602 MB after aggregating seeds and converting dtypes
63.313602 MB total
63.313602 MB after enforcing dtypes
Elapsed time: 613.5132 seconds
Elapsed time: 0:10:13.513432
3724000 rows


,age_group,artifact_path,event_year,input_draw,scenario,sex,value,location
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,baseline,Female,0.0,Japan
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,baseline,Male,0.0,Japan
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,slow_testing_zero_impact_treatment,Female,0.0,Japan
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,slow_testing_zero_impact_treatment,Male,0.0,Japan
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,intermediate_testing_zero_impact_treatment,Female,0.0,Japan


In [25]:
filename = 'transition_count_alzheimers_disease_and_other_dementias'
short_name = 'transitions_ad'

with Timer():
    results.load_measure_from_batch_runs(filename, short_name, filters=loading.FINAL_RESULTS_FILTERS[filename])

print(len(results.data[short_name]), 'rows')
results.data[short_name].head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


101292.812922 MB after read_parquet


655.430949 MB after aggregating seeds and converting dtypes
655.430949 MB total
655.430949 MB after enforcing dtypes
Elapsed time: 2263.3143 seconds
Elapsed time: 0:37:43.314618
29792000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,treatment_initiation_year,value,location
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,transition_count,baseline,Female,alzheimers_blood_based_biomarker_state_to_alzh...,2040,0.0,Japan
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,transition_count,baseline,Female,alzheimers_blood_based_biomarker_state_to_alzh...,2050,0.0,Japan
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,transition_count,baseline,Female,alzheimers_blood_based_biomarker_state_to_alzh...,2060,0.0,Japan
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,transition_count,baseline,Female,alzheimers_blood_based_biomarker_state_to_alzh...,other,0.0,Japan
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,transition_count,baseline,Female,alzheimers_mild_cognitive_impairment_state_to_...,2040,0.0,Japan


In [26]:
incidence_counts = results.process_incidence(
    results.data.incidence_bbbm, results.data.transitions_ad)
print(len(incidence_counts), 'rows')
print_memory_usage(incidence_counts)
incidence_counts

655.430949 MB raw transitions


156.414532 MB after marginalizing treatment


1549.190532 MB after state extraction


['alzheimers_mild_cognitive_impairment_state' 'alzheimers_disease_state']


11172000 rows
301.652483 MB 


,age_group,artifact_path,event_year,input_draw,scenario,sex,value,location,to_state,measure,metric,disease_stage
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,baseline,Female,0.0,Japan,alzheimers_blood_based_biomarker_state,Incidence,Number,alzheimers_blood_based_biomarker_state
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,baseline,Male,0.0,Japan,alzheimers_blood_based_biomarker_state,Incidence,Number,alzheimers_blood_based_biomarker_state
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,slow_testing_zero_impact_treatment,Female,0.0,Japan,alzheimers_blood_based_biomarker_state,Incidence,Number,alzheimers_blood_based_biomarker_state
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,slow_testing_zero_impact_treatment,Male,0.0,Japan,alzheimers_blood_based_biomarker_state,Incidence,Number,alzheimers_blood_based_biomarker_state
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,2025,11,intermediate_testing_zero_impact_treatment,Female,0.0,Japan,alzheimers_blood_based_biomarker_state,Incidence,Number,alzheimers_blood_based_biomarker_state
...,...,...,...,...,...,...,...,...,...,...,...,...
7447995,95_plus,/mnt/team/simulation_science/pub/models/vivari...,2100,243,intermediate_testing_with_treatment,Male,6508.0,Israel,alzheimers_disease_state,Incidence,Number,alzheimers_disease_state
7447996,95_plus,/mnt/team/simulation_science/pub/models/vivari...,2100,243,accelerated_testing_with_treatment,Female,7529.0,Israel,alzheimers_mild_cognitive_impairment_state,Incidence,Number,alzheimers_mild_cognitive_impairment_state
7447997,95_plus,/mnt/team/simulation_science/pub/models/vivari...,2100,243,accelerated_testing_with_treatment,Female,10074.0,Israel,alzheimers_disease_state,Incidence,Number,alzheimers_disease_state
7447998,95_plus,/mnt/team/simulation_science/pub/models/vivari...,2100,243,accelerated_testing_with_treatment,Male,4822.0,Israel,alzheimers_mild_cognitive_impairment_state,Incidence,Number,alzheimers_mild_cognitive_impairment_state


In [27]:
with Timer():
    # incidence_output = results.summarize_and_beautify(incidence_counts, 'to_state')
    # process_* returns event_year as an unordered categorical, and the
    # event_year >= 2025 filter in do_final_processing cannot compare it
    # with the installed pandas version, so cast it back to int16 first
    incidence_output = results.do_final_processing(incidence_counts.astype({'event_year': 'int16'}), include_draws=True)
print_memory_usage(incidence_output)
print(len(incidence_output), 'rows')
incidence_output.tail()

2026-11-19 22:53:32.823309


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 664.7208 seconds


2026-11-19 23:05:39.444395
Elapsed time: 0:12:06.671959
333.250884 MB 
1436400 rows


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:994: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
1436395,2100,Israel,65_plus,Male,MCI due to AD,"Intermediate Testing, With Treatment",Incidence,Rate,0.004873,0.004410,...,0.004669,0.004645,0.005161,0.005022,0.004530,0.005163,0.005199,0.004678,0.005417,0.004616
1436396,2100,Israel,65_plus,Both,MCI due to AD,"Intermediate Testing, With Treatment",Incidence,Rate,0.006304,0.005733,...,0.006628,0.006059,0.006484,0.006490,0.005767,0.006242,0.006732,0.006431,0.006344,0.006062
1436397,2100,Israel,65_plus,Female,MCI due to AD,"Accelerated Testing, With Treatment",Incidence,Rate,0.007575,0.006878,...,0.008368,0.007305,0.007659,0.007796,0.006863,0.007206,0.008096,0.007973,0.007163,0.007349
1436398,2100,Israel,65_plus,Male,MCI due to AD,"Accelerated Testing, With Treatment",Incidence,Rate,0.004873,0.004410,...,0.004669,0.004645,0.005161,0.005022,0.004530,0.005163,0.005199,0.004678,0.005417,0.004616
1436399,2100,Israel,65_plus,Both,MCI due to AD,"Accelerated Testing, With Treatment",Incidence,Rate,0.006304,0.005733,...,0.006628,0.006059,0.006484,0.006490,0.005767,0.006242,0.006732,0.006431,0.006344,0.006062


In [28]:
# Save incidence results
save_csv(incidence_output, 'incidence.csv')

In [29]:
!date

Thu Nov 19 23:06:14 PST 2026
